# PoseNeRF End-to-End
A notebook-first tour of the modular reconstruction, localization, and Studio workflow. **DEMO** reads saved artifacts only; **RESEARCH** actions are explicit opt-ins.

Verified: hierarchical rendering, training-depth PnP, frozen NeRF-depth PnP, and Studio navigation. Experimental: iNeRF-inspired refinement and NeRF-depth localization beyond the measured-depth baseline.

In [ ]:
from pathlib import Path
import json, platform, subprocess, sys, time, urllib.request
import torch
from IPython.display import IFrame, Image, display
MODE = 'DEMO'  # Change to RESEARCH only for opt-in expensive actions
ROOT = Path.cwd(); WORKSPACE = ROOT / 'workspace'
assert (ROOT / 'posenerf').is_dir(), 'Run from the repository root'
print({'python': platform.python_version(), 'torch': torch.__version__, 'cuda': torch.cuda.is_available(), 'device': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu', 'dedicated_vram_bytes': torch.cuda.get_device_properties(0).total_memory if torch.cuda.is_available() else 0, 'mode': MODE})

## Configuration
Paths are repository-relative. E3 is `m3_e3_capacity_8x192_12k`; E6 is `m5b_e6_capacity_8x384_12k`. The notebook does not retrain by default.

In [ ]:
RUN_E3 = 'm3_e3_capacity_8x192_12k'
RUN_E6 = 'm5b_e6_capacity_8x384_12k'
M11 = WORKSPACE / 'outputs' / 'm11_official_test_20261105' / 'm11_report.json'
EVIDENCE = WORKSPACE / 'outputs' / 'github_evidence'
report = json.loads(M11.read_text())
print('Official frozen queries:', report['queries_evaluated'], 'complete:', report['complete'])

## Dataset, architecture, and checkpoint identity
Use `posenerf.data` and `posenerf.model` directly; training RGB-D/known poses are used only for reconstruction/reference geometry. Query inference is RGB-only.

In [ ]:
from posenerf.milestone3 import _safe_run, _checkpoint_for, load_run_config
from posenerf.model import HierarchicalNeRF
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
run = _safe_run(WORKSPACE, RUN_E3); config = load_run_config(run)
checkpoint, payload = _checkpoint_for(run, 'hierarchical_nerf_best.pth', config, device)
model = HierarchicalNeRF(config).to(device); model.load_state_dict(payload['model_state_dict']); model.eval()
print({'checkpoint': str(checkpoint.relative_to(ROOT)), 'step': payload.get('step'), 'parameters': sum(p.numel() for p in model.parameters()), 'samples': [config.train.num_coarse_samples, config.train.num_fine_samples]})

## Official held-out localization benchmark
Metrics are loaded from the immutable M11 report; the 2,000-query benchmark is never rerun automatically.

In [ ]:
for method in ('D0','D1_E3','D2_E6'):
    m = report[method]['metrics']; print(method, 'translation median cm=', 100*m['translation_m']['median'], 'rotation median deg=',m['rotation_deg']['median'], '5cm/5deg=',m['success_5cm_5deg'], 'PnP=',report[method]['pnp_success'])
for name in ('benchmark_comparison.png','cumulative_translation_success.png','translation_error_distribution.png','rotation_error_distribution.png','m9_pnp_inlier_matches.png'):
    path = EVIDENCE / name
    if path.exists(): display(Image(filename=str(path)))

## RGB-only localization and iNeRF refinement
`posenerf.milestone9` implements SIFT 2D–3D matching and PnP. `posenerf.milestone9_extension` is an opt-in geometry-guided iNeRF diagnostic. It can reduce RGB residual while worsening pose error, so it is not a promoted method. Query GT pose/depth must remain evaluation-only.

## Fresh one-view reconstruction and localization
This cell is the executable DEMO. It renders one known validation pose (clearly not RGB-only inference), then runs RGB-only retrieval plus E3 NeRF-depth PnP. GT pose is read only for the final evaluation fields.

In [ ]:
LIVE = WORKSPACE / 'outputs' / 'github_evidence' / 'live_demo'
if not LIVE.exists():
    subprocess.run([sys.executable, '-m', 'posenerf.live_demo', '--workspace', 'workspace', '--output', 'workspace/outputs/github_evidence/live_demo'], check=True)
live = json.loads((LIVE/'live_demo_report.json').read_text())
print(live['known_pose_render'])
print(live['rgb_only_pnp'])
display(Image(filename=str(LIVE/'known_pose_e3_render.png')))
display(Image(filename=str(LIVE/'e3_nerf_depth_pnp_matches.png')))
display(Image(filename=str(LIVE/'query_reference.png')))

In [ ]:
# RESEARCH-only examples (intentionally not executed):
# python -m posenerf.milestone9 --workspace workspace --run-name m3_e3_capacity_8x192_12k --cache workspace/outputs/m9_geometry_reference_e3_20261105 --output workspace/outputs/new_m9_run
# python -m posenerf.milestone9_extension --workspace workspace --cache workspace/outputs/m9_geometry_reference_e3_20261105 --output workspace/outputs/new_hybrid_run
assert MODE in {'DEMO','RESEARCH'}

## Interactive PoseNeRF Studio
This reuses `posenerf.studio`. It launches only if the requested local URL is unavailable; stopping is limited to the subprocess created by this cell.

In [ ]:
STUDIO_URL = 'http://127.0.0.1:8772'
studio_process = None
def studio_running(url=STUDIO_URL):
    try: urllib.request.urlopen(url, timeout=1); return True
    except OSError: return False
def start_studio():
    global studio_process
    if studio_running(): return STUDIO_URL
    studio_process = subprocess.Popen([sys.executable, '-m', 'posenerf.studio', '--outputs-dir', str(WORKSPACE/'outputs'), '--host','127.0.0.1','--port','8772'])
    for _ in range(30):
        if studio_running(): return STUDIO_URL
        time.sleep(.5)
    raise RuntimeError('Studio did not become ready')
def stop_notebook_studio():
    global studio_process
    if studio_process is not None and studio_process.poll() is None: studio_process.terminate()
    studio_process = None
print('Studio URL:', STUDIO_URL, 'running:', studio_running())
# display(IFrame(start_studio(), width='100%', height=700))  # opt-in; browser-link fallback is STUDIO_URL

## Conclusions
On held-out Chess, measured training-depth PnP remains preferred. E3 NeRF-depth PnP is a useful experimental geometry branch but underperforms D0 in the official 2,000-frame protocol; E6 is weaker. Results are limited to Chess and should not be interpreted as general state-of-the-art claims.